In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [12]:
data_path = Path('../../data/train_clean.csv')
# keep_default_na=False desactiva la interpretación automática de nulos
# na_values=[] lista vacía — ningún valor adicional se interpreta como nulo
df = pd.read_csv(data_path, keep_default_na=False, na_values=[''])

print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Nulos totales: {df.isnull().sum().sum()}')

Filas: 1460
Columnas: 81
Nulos totales: 0


# Variables Seleccionadas para el Modelo
 
## Variables Numéricas Seleccionadas
Variables que se usan directamente sin transformación adicional.

| Variable | Descripción | Correlación con SalePrice |
|---|---|---|
| OverallQual | Calidad general de materiales y acabados (1-10) | 0.79 |
| GrLivArea | Área habitable sobre el suelo en pies cuadrados | 0.71 |
| GarageArea | Área del garaje en pies cuadrados | 0.62 |
| TotalBsmtSF | Área total del sótano en pies cuadrados | 0.61 |
| FullBath | Número de baños completos sobre el suelo | 0.56 |
| YearBuilt | Año de construcción original | 0.52 |
| YearRemodAdd | Año de última remodelación | 0.51 |
| MasVnrArea | Área de revestimiento de mampostería en pies cuadrados | 0.48 |
| Fireplaces | Número de chimeneas | 0.47 |
| BsmtFinSF1 | Área terminada del sótano tipo 1 en pies cuadrados | 0.39 |
| LotFrontage | Pies lineales de calle conectados a la propiedad | 0.35 |
| WoodDeckSF | Área de terraza de madera en pies cuadrados | 0.32 |
| 2ndFlrSF | Área del segundo piso en pies cuadrados | 0.32 |
| OpenPorchSF | Área de porche abierto en pies cuadrados | 0.32 |
| HalfBath | Número de medios baños sobre el suelo | 0.28 |

**Variables eliminadas por multicolinealidad:**
- `GarageCars` → redundante con `GarageArea`
- `TotRmsAbvGrd` → redundante con `GrLivArea`
- `GarageYrBlt` → redundante con `YearBuilt`
- `1stFlrSF` → redundante con `TotalBsmtSF`

---

## Variables Categóricas — Ordinal Encoding
Variables con jerarquía clara entre categorías. Se asigna un valor 
numérico respetando el orden de menor a mayor calidad o condición.

| Variable | Descripción | Orden |
|---|---|---|
| ExterQual | Calidad de materiales del exterior | Po < Fa < TA < Gd < Ex |
| KitchenQual | Calidad de la cocina | Po < Fa < TA < Gd < Ex |
| BsmtQual | Calidad del sótano (altura) | None < Po < Fa < TA < Gd < Ex |
| HeatingQC | Calidad del sistema de calefacción | Po < Fa < TA < Gd < Ex |
| BsmtExposure | Exposición del sótano al exterior | None < No < Mn < Av < Gd |
| BsmtFinType1 | Calidad del área terminada del sótano | None < Unf < LwQ < Rec < BLQ < ALQ < GLQ |
| GarageFinish | Acabado interior del garaje | None < Unf < RFn < Fin |
| PavedDrive | Tipo de entrada vehicular | N < P < Y |
| LotShape | Forma general del lote | IR3 < IR2 < IR1 < Reg |

---

## Variables Categóricas — One-Hot Encoding
Variables nominales sin orden jerárquico entre categorías. Se crean 
columnas binarias independientes por cada categoría para evitar 
introducir relaciones matemáticas artificiales.

| Variable | Descripción | # Categorías |
|---|---|---|
| Foundation | Tipo de cimentación | 6 |
| GarageType | Ubicación del garaje | 6 |
| MSZoning | Clasificación de zonificación general | 5 |
| SaleCondition | Condición de la venta | 6 |

---

## Variables Categóricas — Binary Encoding
Variables con exactamente 2 categorías. Se codifican como 0 y 1 
sin necesidad de One-Hot Encoding.

| Variable | Descripción | Codificación |
|---|---|---|
| CentralAir | Aire acondicionado central | N=0, Y=1 |

---

## Variables Categóricas — Ordinal Encoding Agrupado
Variables con alta cardinalidad que requieren agrupación previa 
por precio mediano antes de aplicar Ordinal Encoding.

| Variable | Descripción | # Categorías originales |
|---|---|---|
| Neighborhood | Ubicación dentro de Ames | 25 → 5 segmentos |

**Segmentos de Neighborhood por precio mediano:**
| Segmento | Vecindarios | Precio mediano aprox. |
|---|---|---|
| 1 - Muy bajo | MeadowV, IDOTRR, BrDale | < 110,000 USD |
| 2 - Bajo | OldTown, Edwards, BrkSide | 119,000 - 135,000 USD |
| 3 - Medio | NAmes, Mitchel, SawyerW | 140,000 - 180,000 USD |
| 4 - Alto | CollgCr, NWAmes, Gilbert | 180,000 - 215,000 USD |
| 5 - Premium | NridgHt, NoRidge, StoneBr | > 270,000 USD |

---

## Target
| Variable | Descripción |
|---|---|
| SalePrice_log | Logaritmo natural del precio de venta — target del modelo |